# Tight Junction (TJ) Quantification
### Cellpose-Based Pipeline for 3D Blood–Brain Barrier Confocal Images

---

## Overview

This notebook provides a reproducible, GPU-accelerated pipeline for detecting and quantifying **tight junction (TJ) complexes** in confocal fluorescence images of the **blood–brain barrier (BBB)**. Using the [Cellpose](https://github.com/MouseLand/cellpose) instance segmentation framework, the pipeline applies three sequential morphological filters (size, border, noise) to isolate TJ structures, then exports per-object morphometric measurements — area, circularity, and eccentricity — as a structured CSV.

The pipeline supports:
- **Single 2D images** — direct analysis of a flat grayscale image
- **Single Z-slice** from a 3D confocal stack — user-selected slice index
- **Full Z-stack** — loops every slice, aggregates all results, and exports a unified CSV

> 💡 **Best use of this pipeline:** Process a **single 2D image or one representative Z-slice** at a time.  
> Single-slice analysis is faster, easier to quality-check visually, and yields the most reproducible results for quantitative comparison across samples.  
> Full-stack mode is available for exploratory analysis but results should be interpreted with care — optical aberrations and signal variation across Z can affect segmentation quality.

---
## Before You Begin

1. Click **File → Save a copy in Drive** to run this notebook under your own Google account.  
   This ensures your runtime, GPU quota, and storage are used — not the author's.
2. Set the runtime: **Runtime → Change runtime type → T4 GPU**

> ⚠️ Running without a GPU is possible but significantly slower for Cellpose segmentation.

---
## Installation & Imports

The cell below installs all required Python packages into the Colab session.  
Output is suppressed via `%%capture`; run once per session — no restart needed.

In [ ]:
%%capture
!pip install -U cellpose scikit-image matplotlib pandas

---
## Parameter Configuration

Adjust the values below to match your microscopy setup and experimental design **before** running any pipeline cells.  
All downstream steps inherit these values automatically — no other edits are required.

| Parameter | Default | Description |
|---|---|---|
| `PIXEL_SCALE_UM` | `0.16` | Physical pixel size in µm (XY plane) — check your acquisition metadata |
| `MIN_AREA_PX` | `100` | Minimum object area in pixels; smaller detections are discarded |
| `BORDER_MARGIN_PX` | `50` | Objects whose centroid falls within this many pixels of any edge are discarded |
| `Z_INDEX` | `0` | Default Z-slice fallback when no interactive input is given |
| `CELLPOSE_DIAMETER` | `None` | Expected cell diameter in pixels; `None` = Cellpose auto-detects |
| `NOISE_MAX_AREA_PX` | `300` | Size ceiling for the circular-noise removal step |
| `NOISE_MIN_CIRCULARITY` | `0.80` | Circularity threshold (0–1) above which small objects are treated as noise artifacts |

In [ ]:
# ── 0. IMPORTS ───────────────────────────────────────────────────────────────
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from skimage import io, measure
from cellpose import models
from google.colab import files

# ── 1. CONFIGURATION ─────────────────────────────────────────────────────────
PIXEL_SCALE_UM        = 0.16   # microns per pixel (XY)
MIN_AREA_PX           = 100    # minimum object area to keep (pixels)
BORDER_MARGIN_PX      = 50     # centroid must be at least this far from image edge
Z_INDEX               = 0      # default Z slice (fallback when no user input is given)
CELLPOSE_DIAMETER     = None   # None = auto-detect cell diameter
NOISE_MAX_AREA_PX     = 300    # objects above this size are never removed by the shape filter
NOISE_MIN_CIRCULARITY = 0.80   # circularity (0–1); objects exceeding this are noise candidates

---
## Step 1 — Upload & Load Image

Run the cell below to upload your image file (`.tif`, `.tiff`, or any format supported by `scikit-image`).

After loading, the pipeline automatically detects the image dimensionality:

- **2D image** → proceeds directly to segmentation with no further input required.
- **3D image (Z-stack)** → you will be prompted interactively to choose:
  - **Option 1 — Single slice:** enter the Z index you wish to analyse (recommended).
  - **Option 2 — Full stack:** all slices are processed sequentially and results are aggregated.

In [ ]:
# ── 2. UPLOAD & LOAD ─────────────────────────────────────────────────────────
uploaded = files.upload()
filename = next(iter(uploaded))
base, ext = os.path.splitext(filename)

img_stack = io.imread(filename)
print(f"Loaded : {filename}")
print(f"Shape  : {img_stack.shape}  |  Dtype : {img_stack.dtype}")

# ── Dimensionality Detection & Interactive Mode Selection ────────────────────
if img_stack.ndim == 2:
    print("\n2D image detected — running single-slice analysis directly.")
    slices_to_process = [img_stack]
    slice_labels      = ["2D"]
    mode              = "single"

elif img_stack.ndim == 3:
    n_slices = img_stack.shape[0]
    print(f"\n3D image detected — {n_slices} Z-slice(s) available.")
    print("\n" + "─" * 58)
    print("  Options:")
    print("    [1]  Single slice  (recommended — faster, easier to QC)")
    print("    [2]  Full stack    (all slices, results aggregated)")
    print("─" * 58)
    print("\n  💡 Best practice: choose a single representative slice.")
    print("     Full-stack mode is useful for exploratory analysis\n")

    mode_input = input("Process [1] Single slice  or  [2] Full stack?  Enter 1 or 2: ").strip()

    if mode_input == "2":
        mode              = "stack"
        slices_to_process = [img_stack[i] for i in range(n_slices)]
        slice_labels      = [f"Z{i:03d}" for i in range(n_slices)]
        print(f"\nFull-stack mode selected — will process all {n_slices} slices.")
    else:
        mode    = "single"
        z_input = input(f"Which Z slice? Enter index 0–{n_slices - 1} (default = {Z_INDEX}): ").strip()
        z_idx   = int(z_input) if z_input.isdigit() else Z_INDEX
        z_idx   = max(0, min(z_idx, n_slices - 1))
        slices_to_process = [img_stack[z_idx]]
        slice_labels      = [f"Z{z_idx:03d}"]
        print(f"\nSingle-slice mode selected — using Z index {z_idx}.")

else:
    raise ValueError(f"Unexpected image shape: {img_stack.shape}. Expected 2D or 3D array.")

print(f"\nReady — {len(slices_to_process)} slice(s) queued: {slice_labels}")

---
## Step 2 — Initialise Cellpose Model

The Cellpose `CellposeModel` is loaded once here and reused across all slices.  
GPU availability is detected automatically — the model falls back to CPU if no GPU is found.

In [ ]:
# ── MODEL INITIALISATION ─────────────────────────────────────────────────────
USE_GPU = torch.cuda.is_available()
print(f"GPU available : {USE_GPU}" + (f"  ({torch.cuda.get_device_name(0)})" if USE_GPU else ""))

model = models.CellposeModel(gpu=USE_GPU)
print("Cellpose model loaded and ready.")

---
## Steps 3–9 — Per-Slice Processing Function

The function below encapsulates the complete analysis pipeline for a single 2D image slice:

| Step | Operation | Output |
|---|---|---|
| **3** | Cellpose instance segmentation | Raw object masks |
| **4** | Size filter — remove objects below `MIN_AREA_PX` | `mask_size` |
| **5** | Border filter — remove edge-adjacent objects | `mask_border` |
| **6** | Circular noise filter — remove small, round artefacts | `mask_clean` |
| **7** | Multi-panel visualisation (raw → filtered → final) | Saved `.tif` panel |
| **8** | Final overlay image | Saved `.tif` overlay |
| **9** | Morphometric measurement (area, circularity, eccentricity) | `pandas.DataFrame` |

The function returns a measurement DataFrame and the overlay file path for downstream aggregation.

In [ ]:
# ── PIPELINE FUNCTION ─────────────────────────────────────────────────────────
def process_slice(img, slice_label, base):
    """
    Run the full TJ quantification pipeline on a single 2D image slice.

    Parameters
    ----------
    img         : 2-D numpy array  — the image slice to analyse
    slice_label : str              — label for this slice (e.g. 'Z003' or '2D')
    base        : str              — base filename (without extension) for outputs

    Returns
    -------
    df           : pandas DataFrame of per-object morphometric measurements
    overlay_path : str — path to the saved final-overlay TIFF
    """
    print(f"\n{'='*62}")
    print(f"  Slice : {slice_label}  |  Shape : {img.shape}")
    print(f"{'='*62}\n")

    # ── Step 3: Cellpose segmentation ─────────────────────────────────────────
    print("── Step 3 : Cellpose Segmentation ──────────────────────────────────")
    masks, flows, styles = model.eval(img, diameter=CELLPOSE_DIAMETER, do_3D=False)
    n_raw = int(masks.max())
    print(f"   Complete  →  {n_raw} raw objects detected.")

    # ── Step 4: Size filter ────────────────────────────────────────────────────
    print("\n── Step 4 : Size Filter (min area = {MIN_AREA_PX} px) ──────────────────")
    mask_size = np.zeros_like(masks)
    for prop in measure.regionprops(masks):
        if prop.area >= MIN_AREA_PX:
            mask_size[masks == prop.label] = prop.label
    n_size = len(np.unique(mask_size)) - 1
    print(f"   Kept : {n_size}   Removed : {n_raw - n_size}")

    # ── Step 5: Border filter ──────────────────────────────────────────────────
    print(f"\n── Step 5 : Border Filter (margin = {BORDER_MARGIN_PX} px) ─────────────────")
    h, w = mask_size.shape
    mask_border = np.zeros_like(mask_size)
    for prop in measure.regionprops(mask_size):
        y, x = prop.centroid
        if min(y, x, h - y, w - x) >= BORDER_MARGIN_PX:
            mask_border[mask_size == prop.label] = prop.label
    n_border = len(np.unique(mask_border)) - 1
    print(f"   Kept : {n_border}   Removed : {n_size - n_border}")

    # ── Step 6: Circular noise filter ─────────────────────────────────────────
    print(f"\n── Step 6 : Circular Noise Filter (area < {NOISE_MAX_AREA_PX} px & circ > {NOISE_MIN_CIRCULARITY}) ──")
    mask_clean = np.zeros_like(mask_border)
    n_noise    = 0
    for prop in measure.regionprops(mask_border):
        circ = (4 * np.pi * prop.area) / (prop.perimeter ** 2) if prop.perimeter > 0 else 0
        if prop.area < NOISE_MAX_AREA_PX and circ > NOISE_MIN_CIRCULARITY:
            n_noise += 1
            continue
        mask_clean[mask_border == prop.label] = prop.label
    n_final  = len(np.unique(mask_clean)) - 1
    pct_diff = ((n_raw - n_final) / n_raw * 100) if n_raw > 0 else 0.0
    print(f"   Kept : {n_final}   Noise removed : {n_noise}")
    print(f"   Total removed across all filters : {n_raw - n_final}  ({pct_diff:.1f}% of raw)")

    # ── Step 7: Multi-panel visualisation ─────────────────────────────────────
    print("\n── Step 7 : Multi-Panel Visualisation ──────────────────────────────")
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(img, cmap="gray")
    axes[0].imshow(masks, cmap="nipy_spectral", alpha=0.5)
    axes[0].set_title(f"1. Raw Segmentation\n{n_raw} objects", fontsize=12)
    axes[0].axis("off")

    axes[1].imshow(img, cmap="gray")
    axes[1].imshow(mask_border, cmap="nipy_spectral", alpha=0.5)
    axes[1].set_title(
        f"2. Size + Border Filtered\n{n_border} objects  [\u2212{n_raw - n_border} from raw]",
        fontsize=12
    )
    axes[1].axis("off")

    axes[2].imshow(img, cmap="gray")
    axes[2].imshow(mask_clean, cmap="nipy_spectral", alpha=0.5)
    axes[2].set_title(
        f"3. Noise Filtered\n{n_final} objects  [\u2212{n_raw - n_final} from raw]",
        fontsize=12
    )
    axes[2].axis("off")

    plt.suptitle(
        f"{base}  |  Slice: {slice_label}  |  "
        f"min_area={MIN_AREA_PX}px  border={BORDER_MARGIN_PX}px  "
        f"noise<{NOISE_MAX_AREA_PX}px & circ>{NOISE_MIN_CIRCULARITY}",
        fontsize=9, y=1.01
    )
    plt.tight_layout()
    panel_path = f"{base}_{slice_label}_pipeline_overview.tif"
    plt.savefig(panel_path, bbox_inches="tight", dpi=300)
    plt.show()
    print(f"   Panel saved  \u2192  {panel_path}")

    # ── Step 8: Final overlay ──────────────────────────────────────────────────
    print("\n── Step 8 : Final Overlay ───────────────────────────────────────────")
    fig2, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img, cmap="gray")
    ax.imshow(mask_clean, cmap="nipy_spectral", alpha=0.5)
    ax.set_title(f"Final Segmentation Overlay  \u2014  {slice_label}  ({n_final} objects)", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    overlay_path = f"{base}_{slice_label}_tj_overlay.tif"
    plt.savefig(overlay_path, bbox_inches="tight", dpi=300)
    plt.show()
    print(f"   Overlay saved  \u2192  {overlay_path}")

    # ── Step 9: Morphometric measurements ─────────────────────────────────────
    print("\n── Step 9 : Morphometric Measurements ──────────────────────────────")
    pixel_area_um2 = PIXEL_SCALE_UM ** 2
    records = []
    for prop in measure.regionprops(mask_clean):
        circ = (4 * np.pi * prop.area) / (prop.perimeter ** 2) if prop.perimeter > 0 else 0
        records.append({
            "Slice":         slice_label,
            "Object_ID":     prop.label,
            "Area_Pixels":   prop.area,
            "Area_Microns2": round(prop.area * pixel_area_um2, 4),
            "Circularity":   round(circ, 4),
            "Eccentricity":  round(prop.eccentricity, 4),
            "Major_Axis_px": round(prop.axis_major_length, 2),
            "Minor_Axis_px": round(prop.axis_minor_length, 2),
            "Centroid_Y":    round(prop.centroid[0], 2),
            "Centroid_X":    round(prop.centroid[1], 2),
        })
    df = pd.DataFrame(records)

    if len(df) > 0:
        print(f"\n   Statistics for {slice_label}:")
        print(f"     Total TJ objects : {len(df)}")
        print(f"     Area  (px)       : {df['Area_Pixels'].mean():.1f} \u00b1 {df['Area_Pixels'].std():.1f}"
              f"  [{df['Area_Pixels'].min():.0f} \u2013 {df['Area_Pixels'].max():.0f}]")
        print(f"     Area  (\u00b5m\u00b2)      : {df['Area_Microns2'].mean():.2f} \u00b1 {df['Area_Microns2'].std():.2f}")
        print(f"     Circularity      : {df['Circularity'].mean():.3f} \u00b1 {df['Circularity'].std():.3f}")
        print(f"     Eccentricity     : {df['Eccentricity'].mean():.3f} \u00b1 {df['Eccentricity'].std():.3f}")
    else:
        print(f"   No objects remaining after filtering for slice {slice_label}.")

    return df, overlay_path

---
## Run Analysis

The cell below executes the pipeline for every queued slice and collects all results.

- **Single-slice mode** → one pass, one output DataFrame.
- **Full-stack mode** → iterates over all Z-slices, prints per-slice statistics, then displays an aggregate summary across the entire stack.

In [ ]:
# ── RUN PIPELINE ─────────────────────────────────────────────────────────────
all_dfs      = []
all_overlays = []

for img_slice, slice_label in zip(slices_to_process, slice_labels):
    df_slice, overlay_path = process_slice(img_slice, slice_label, base)
    all_dfs.append(df_slice)
    all_overlays.append(overlay_path)

print(f"\n{'='*62}")
print(f"  Pipeline complete  \u2014  {len(all_dfs)} slice(s) processed.")
print(f"{'='*62}")

# ── Aggregate Summary (stack mode) ───────────────────────────────────────────
if mode == "stack" and len(all_dfs) > 1:
    print("\n── Full-Stack Aggregate Summary ─────────────────────────────────────")
    summary_rows = []
    for label, df_s in zip(slice_labels, all_dfs):
        if len(df_s) > 0:
            summary_rows.append({
                "Slice":             label,
                "Total_Objects":     len(df_s),
                "Avg_Area_px":       round(df_s["Area_Pixels"].mean(), 1),
                "Avg_Area_um2":      round(df_s["Area_Microns2"].mean(), 3),
                "Std_Area_um2":      round(df_s["Area_Microns2"].std(), 3),
                "Avg_Circularity":   round(df_s["Circularity"].mean(), 4),
                "Avg_Eccentricity":  round(df_s["Eccentricity"].mean(), 4),
            })
    stack_summary = pd.DataFrame(summary_rows)
    print(stack_summary.to_string(index=False))

    full_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n── Grand Totals (all {len(all_dfs)} slices) ──────────────────────────────")
    print(f"   Total objects across all slices : {len(full_df)}")
    print(f"   Area (\u00b5m\u00b2)      : {full_df['Area_Microns2'].mean():.2f} \u00b1 {full_df['Area_Microns2'].std():.2f}")
    print(f"   Circularity      : {full_df['Circularity'].mean():.3f} \u00b1 {full_df['Circularity'].std():.3f}")
    print(f"   Eccentricity     : {full_df['Eccentricity'].mean():.3f} \u00b1 {full_df['Eccentricity'].std():.3f}")
else:
    full_df = all_dfs[0] if all_dfs else pd.DataFrame()

---
## Step 10 — Export & Download Results

The cell below writes a single CSV file containing:
1. **Header metadata** — filename, mode, number of slices, all filter parameters.
2. **Per-slice summary table** — one row per slice with aggregate statistics.
3. **Per-object data table** — one row per detected TJ object across all slices.

All overlay TIFF images and the CSV are downloaded automatically to your local machine.

In [ ]:
# ── 10. EXPORT & DOWNLOAD ────────────────────────────────────────────────────
csv_path = f"{base}_tj_measurements.csv"

# Per-slice summary
summary_rows = []
for label, df_s in zip(slice_labels, all_dfs):
    if len(df_s) > 0:
        summary_rows.append({
            "Slice":                  label,
            "Total_Objects":          len(df_s),
            "Avg_Area_Pixels":        round(df_s["Area_Pixels"].mean(), 3),
            "Std_Area_Pixels":        round(df_s["Area_Pixels"].std(), 3),
            "Avg_Area_Microns2":      round(df_s["Area_Microns2"].mean(), 3),
            "Std_Area_Microns2":      round(df_s["Area_Microns2"].std(), 3),
            "Avg_Circularity":        round(df_s["Circularity"].mean(), 4),
            "Avg_Eccentricity":       round(df_s["Eccentricity"].mean(), 4),
            "PIXEL_SCALE_UM":         PIXEL_SCALE_UM,
            "MIN_AREA_PX":            MIN_AREA_PX,
            "BORDER_MARGIN_PX":       BORDER_MARGIN_PX,
            "NOISE_MAX_AREA_PX":      NOISE_MAX_AREA_PX,
            "NOISE_MIN_CIRCULARITY":  NOISE_MIN_CIRCULARITY,
        })
summary_df = pd.DataFrame(summary_rows)

# Write CSV with sections
with open(csv_path, "w") as f:
    f.write(f"# TJ QUANTIFICATION RESULTS\n")
    f.write(f"# Source file : {filename}\n")
    f.write(f"# Mode        : {mode}  |  Slices processed : {len(slice_labels)}\n")
    f.write(f"# Slice labels: {slice_labels}\n")
    f.write("#\n# SUMMARY (one row per slice)\n")
summary_df.to_csv(csv_path, mode="a", index=False)

with open(csv_path, "a") as f:
    f.write("\n# PER-OBJECT DATA (one row per detected TJ object)\n")
full_df.to_csv(csv_path, mode="a", index=False)

print(f"CSV saved  \u2192  {csv_path}")
print(f"Rows in per-object table : {len(full_df)}")
print("\nStarting downloads...")

files.download(csv_path)
print(f"  Downloaded : {csv_path}")

for overlay in all_overlays:
    files.download(overlay)
    print(f"  Downloaded : {overlay}")

print(f"\nAll {1 + len(all_overlays)} file(s) downloaded successfully.")